# CE541E08 — Unit 4 · Day 29 — Loading, Inspecting and Filtering Data

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 29 of 45 |
| **CO** | CO4 |
| **Topics** | Building large DataFrames · .info() · .describe() · .value_counts() · filtering · .nlargest() |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 29"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Working with Larger Datasets

In real hydrology work, DataFrames are large — months of daily data, years of monthly records, dozens of stations. Today we build a 20-year monthly streamflow DataFrame (240 records) and learn how to inspect, summarise, and filter it efficiently.

---
## Code Block 1 — Building a Large DataFrame

### What this code does

We generate a 20-year × 12-month streamflow dataset (240 records) programmatically, then create a DataFrame with columns: Date, Flow_m3s, and QA_Flag.

### Why each step is taken

**`np.tile(base, 20)`:**
`np.tile` repeats an array a specified number of times. `tile(base, 20)` stacks 20 copies of the 12-element base array end-to-end, giving a 240-element array — one value per month for 20 years. This is much cleaner than writing the same list 20 times.

**`np.random.normal(0, base*0.2, 240)`:**
Adds realistic year-to-year variation. The standard deviation is 20% of the base value — so a month with a base of 456 m³/s might vary by ±91 m³/s. `np.maximum(..., 5)` ensures no unrealistically negative or zero flows.

**`np.where(flows<10,'LOW', np.where(flows>800,'HIGH','GOOD'))`:**
Nested `np.where` — the inner one flags HIGH records first, then the outer one flags LOW records. Everything else gets 'GOOD'. This creates a realistic QA flag column.

### Algorithm

```
1. Generate date labels: "2005-Jan", "2005-Feb", ..., "2024-Dec"
   list comprehension: for each year, for each month

2. np.tile(base, 20) → repeat 12-month pattern 20 times → (240,)
   + np.random.normal → add ±20% noise
   np.maximum(...,5) → prevent non-physical values

3. np.where to assign QA flags:
   flow < 10  → 'LOW'
   flow > 800 → 'HIGH'
   otherwise  → 'GOOD'

4. pd.DataFrame({'Date':..., 'Flow_m3s':..., 'QA_Flag':...})
```

### Expected output

```
Shape: (240, 3)
         Date  Flow_m3s QA_Flag
0    2005-Jan      36.9    GOOD
1    2005-Feb      38.0    GOOD
2    2005-Mar      28.2    GOOD
3    2005-Apr      26.7    GOOD
4    2005-May      30.4    GOOD
5    2005-Jun     252.3    GOOD
6    2005-Jul     469.3    GOOD
7    2005-Aug     381.9    GOOD
```

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
years  = list(range(2005, 2025))
months = ['Jan','Feb','Mar','Apr','May','Jun',
          'Jul','Aug','Sep','Oct','Nov','Dec']

# Date labels: one per month-year combination
dates = [f"{y}-{m}" for y in years for m in months]   # 240 labels

# Base seasonal pattern — repeated 20 times using np.tile
base  = np.array([45, 38, 28, 22, 35, 234, 456, 389, 198, 89, 62, 50])
flows = np.round(
    np.maximum(np.tile(base, 20) + np.random.normal(0, base*0.2, 240), 5), 1
)

# Nested np.where to assign QA flags
flags = np.where(flows < 10, 'LOW',
        np.where(flows > 800, 'HIGH', 'GOOD'))

df = pd.DataFrame({'Date':dates, 'Flow_m3s':flows, 'QA_Flag':flags})
print(f"Shape: {df.shape}")
print(df.head(8))

### 🔁 Try this

Change `np.random.seed(42)` to `np.random.seed(0)` and re-run.

- Does the shape change?
- How many HIGH records are there now compared to before?

---
## Code Block 2 — .info() and .describe()

### What this code does

We call `.info()` and `.describe()` on the 240-record DataFrame, then use `.value_counts()` to count the QA flag distribution.

### Why each step is taken

**`df.info()`:**
Prints the number of rows, column names, count of non-null values per column, and memory usage. This is the first thing to run on a new dataset — it immediately reveals missing values (non-null count < total rows) and wrong dtypes.

**`df.describe()`:**
Computes descriptive statistics for all numeric columns automatically. Returns count, mean, std, min, Q25, median (50%), Q75, and max. One call replaces eight separate calculations.

**`df['QA_Flag'].value_counts()`:**
Counts occurrences of each unique value in a categorical column. Essential for understanding data quality — how many GOOD, HIGH, LOW, MISSING records are there?

### Algorithm

```
1. df.info()     → column names, dtypes, null counts, memory
2. df.describe() → stats for numeric columns (Flow_m3s only here)
3. df['QA_Flag'].value_counts() → count per category
```

### Expected output

```
=== info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
...

=== describe() ===
       Flow_m3s
count  240.000
mean   167.8...
...

=== value_counts ===
QA_Flag
GOOD    221
HIGH     19
LOW       0
```

In [ ]:
import pandas as pd, numpy as np
np.random.seed(42)
years=list(range(2005,2025)); months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dates=[f"{y}-{m}" for y in years for m in months]
base=np.array([45,38,28,22,35,234,456,389,198,89,62,50])
flows=np.round(np.maximum(np.tile(base,20)+np.random.normal(0,base*0.2,240),5),1)
flags=np.where(flows<10,'LOW',np.where(flows>800,'HIGH','GOOD'))
df=pd.DataFrame({'Date':dates,'Flow_m3s':flows,'QA_Flag':flags})

# .info(): column names, dtypes, non-null counts, memory usage
print("=== info() ==="); df.info()
print()
# .describe(): statistics for all numeric columns
print("=== describe() ==="); print(df.describe())
print()
# .value_counts(): frequency of each unique value in a column
print("=== value_counts ==="); print(df['QA_Flag'].value_counts())

### 🔁 Try this

Run `df['Flow_m3s'].describe()` and compare with `df.describe()`.

What is the difference? When would you use each?

---
## Code Block 3 — Filtering Rows

### What this code does

We filter the DataFrame using boolean conditions on columns — selecting HIGH records, records above 300 m³/s, and using `.nlargest()` to find peak events.

### Why each step is taken

**`df[df['QA_Flag']=='HIGH']`:**
`df['QA_Flag']=='HIGH'` produces a True/False Series of 240 values. Passing it back into `df[...]` keeps only the rows where True. This is the Pandas equivalent of NumPy boolean indexing.

**`df[df['Flow_m3s']>300]`:**
Same pattern — keeps only rows where flow exceeds 300 m³/s.

**`.mean()` on filtered DataFrame:**
After filtering to a subset of rows, all column methods work on that subset. `.mean()` gives the mean of only the high-flow months.

**`df.nlargest(5,'Flow_m3s')`:**
Returns the 5 rows with the largest values in the specified column. More convenient than sorting and slicing. `.nsmallest()` is the complement.

### Algorithm

```
1. high = df[df['QA_Flag']=='HIGH']
   → keeps only rows where flag is HIGH
   len(high) → count

2. above_300 = df[df['Flow_m3s']>300]
   → keeps only rows where flow > 300
   above_300['Flow_m3s'].mean() → mean of filtered flows

3. df.nlargest(5,'Flow_m3s') → top 5 peak months
   df.nsmallest(5,'Flow_m3s') → 5 lowest months
```

### Expected output

```
Column type: <class 'pandas.core.series.Series'>
HIGH records: 19
Flow>300 m3/s: 80 months
Mean high    : 454.6 m3/s
Top 5 peak flows:
  Date  Flow_m3s
  2018-Jul  609.2
  ...
```

In [ ]:
import pandas as pd, numpy as np
np.random.seed(42)
years=list(range(2005,2025)); months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dates=[f"{y}-{m}" for y in years for m in months]
base=np.array([45,38,28,22,35,234,456,389,198,89,62,50])
flows=np.round(np.maximum(np.tile(base,20)+np.random.normal(0,base*0.2,240),5),1)
flags=np.where(flows<10,'LOW',np.where(flows>800,'HIGH','GOOD'))
df=pd.DataFrame({'Date':dates,'Flow_m3s':flows,'QA_Flag':flags})

# Selecting a column returns a Series
print("Column type:", type(df['Flow_m3s']))

# Boolean filtering: keep only rows where condition is True
high = df[df['QA_Flag']=='HIGH']
print(f"HIGH records: {len(high)}")

# Filter by numeric threshold
above_300 = df[df['Flow_m3s'] > 300]
print(f"Flow>300 m3/s: {len(above_300)} months")
print(f"Mean high    : {above_300['Flow_m3s'].mean():.1f} m3/s")

# nlargest: top n rows by a column
print("Top 5 peak flows:")
print(df.nlargest(5, 'Flow_m3s').to_string(index=False))
print()
print("5 lowest flows:")
print(df.nsmallest(5, 'Flow_m3s').to_string(index=False))

### 🔁 Try this

Filter for monsoon months only (Jun, Jul, Aug, Sep).

You can check if a date string contains the month name:
`df[df['Date'].str.contains('Jun|Jul|Aug|Sep')]`

- How many monsoon records are there across 20 years?
- What is the mean monsoon flow?

---
## Session Summary — Loading, Inspecting and Filtering

| Method | What it does |
|---|---|
| `df.info()` | Dtypes, null counts, memory usage |
| `df.describe()` | Stats for all numeric columns |
| `df['col'].value_counts()` | Count of each unique value |
| `df[df['col']>x]` | Filter rows by numeric condition |
| `df[df['col']=='val']` | Filter rows by string condition |
| `df['col'].mean()` | Mean of filtered or full column |
| `df.nlargest(n,'col')` | Top n rows by column |
| `df.nsmallest(n,'col')` | Bottom n rows by column |
| `np.tile(arr, n)` | Repeat array n times |

---
## Day 29 Assignment

Using the 240-record DataFrame from Code Block 1:

1. Count how many months had flow > 400 m³/s
2. Compute the mean flow of LOW flag records
3. Compute the fraction of records flagged as HIGH (as a decimal, 3 d.p.)

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np
np.random.seed(42)
years=list(range(2005,2025)); months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dates=[f"{y}-{m}" for y in years for m in months]
base=np.array([45,38,28,22,35,234,456,389,198,89,62,50])
flows=np.round(np.maximum(np.tile(base,20)+np.random.normal(0,base*0.2,240),5),1)
flags=np.where(flows<10,'LOW',np.where(flows>800,'HIGH','GOOD'))
df=pd.DataFrame({'Date':dates,'Flow_m3s':flows,'QA_Flag':flags})

high_months = ???   # count where Flow_m3s > 400
low_mean    = ???   # mean of LOW records
pct_high    = ???   # fraction flagged HIGH (decimal)

print(f"Months flow>400: {high_months}")
print(f"Mean LOW flow  : {low_mean:.2f} m3/s")
print(f"Fraction HIGH  : {pct_high:.3f}")

---
- [ ] Run all cells and verify outputs
- [ ] Complete the assignment cell
- [ ] Upload to GitHub: `Unit4_Pandas/CE541E08_U4_Day29.ipynb`
- [ ] Commit message: `Day 29 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*